# Filament Segmentation 2026 — training driver (Kaggle)

**Setup checklist (once per notebook):**
1. Add data: `+ Add Input` → competition **filament-segmentation-2026**.
2. Add the code: either set `GITHUB_URL` below (public repo), or upload the repo's `src/`, `scripts/`, `configs/` as a private Kaggle Dataset named `filseg-code` and attach it.
3. To **resume** a previous session: attach that notebook's Output as an input and set `RESUME_FROM`.
4. Settings → Accelerator: **GPU T4/P100**; Internet: **On** (needed for pip + pretrained weights).

Each session: builds labels/folds/prep into ephemeral `/kaggle/tmp` (~6 min), trains with a time budget, and persists `checkpoints/` to `/kaggle/working` for the next session.

In [ ]:
from pathlib import Path

def find_input(name):
    """Locate a mounted input dir regardless of Kaggle's mount layout."""
    root = Path("/kaggle/input")
    for pattern in (name, f"*/{name}", f"*/*/{name}", f"*/*/*/{name}"):
        for p in root.glob(pattern):
            if p.is_dir():
                return p
    return None

GITHUB_URL = ""           # optional: clone a repo instead of the filseg-code dataset
FOLD = 0
CONFIG = "configs/baseline.yaml"
MAX_HOURS = 10.5           # stop cleanly before the 12h session limit

# resume continues automatically when the filseg-ckpt dataset is attached
_ck = find_input("filseg-ckpt")
RESUME_FROM = str(_ck) if _ck else ""
print("resume:", RESUME_FROM or "fresh run")

In [ ]:
import os, shutil, subprocess, sys

CODE = Path("/kaggle/tmp/code")
if CODE.exists():
    shutil.rmtree(CODE)
if GITHUB_URL:
    subprocess.run(["git", "clone", "--depth", "1", GITHUB_URL, str(CODE)], check=True)
else:
    src = find_input("filseg-code")
    assert src is not None, f"filseg-code dataset not mounted; /kaggle/input = {sorted(os.listdir('/kaggle/input'))}"
    shutil.copytree(src, CODE)
os.chdir(CODE)
%pip install -q segmentation-models-pytorch albumentations pycocotools
print("code ready at", CODE)

In [ ]:
# derived artifacts live in ephemeral /kaggle/tmp; checkpoints are copied out at the end
os.environ["FILSEG_WORK"] = "/kaggle/tmp/filseg_work"

subprocess.run([sys.executable, "scripts/build_labels.py"], check=True)
subprocess.run([sys.executable, "scripts/make_folds.py"], check=True)
subprocess.run([sys.executable, "scripts/build_prep.py"], check=True)

In [ ]:
# optional resume: locate .pth files whatever the dataset layout and place them
# under <work>/checkpoints/<run_name>/
if RESUME_FROM:
    dest_root = Path(os.environ["FILSEG_WORK"]) / "checkpoints"
    found = sorted(Path(RESUME_FROM).rglob("*.pth"))
    for f in found:
        dest = dest_root / f.parent.name / f.name
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(f, dest)
    if found:
        print("resumed checkpoints:", [str(p.parent.name + "/" + p.name) for p in found])
    else:
        print("WARNING: no .pth files found under", RESUME_FROM)

In [ ]:
import yaml

cfg = yaml.safe_load(open(CONFIG, encoding="utf-8"))
cfg["train"]["max_hours"] = MAX_HOURS
tuned = "/kaggle/tmp/cfg.yaml"
yaml.safe_dump(cfg, open(tuned, "w", encoding="utf-8"))

# smoke-test the encoder/decoder pairing BEFORE spending GPU hours: the encoder
# must expose a real stride-2 stage (no zero-channel pyramid level) and every
# head must produce full-resolution output
sys.path.insert(0, "src")
import torch
from filseg.model import build_model

_m = build_model(cfg)
_enc_ch = _m.net.encoder.out_channels
assert 0 not in _enc_ch, f"encoder has a zero-channel pyramid level (no stride-2 stage): {_enc_ch}"
_y = _m(torch.zeros(1, cfg["model"].get("in_channels", 3), 512, 512))
assert _y["mask"].shape[-2:] == (512, 512), _y["mask"].shape
print("model smoke-test OK, encoder channels:", _enc_ch)
del _m, _y

subprocess.run([sys.executable, "scripts/train_fold.py", "--config", tuned, "--fold", str(FOLD)], check=True)

In [ ]:
# persist checkpoints (and a submission from the current best) to the notebook output
work = Path(os.environ["FILSEG_WORK"])
out_ck = Path("/kaggle/working/checkpoints")
if out_ck.exists():
    shutil.rmtree(out_ck)
shutil.copytree(work / "checkpoints", out_ck)

best = out_ck / f"{cfg['name']}_f{FOLD}" / "best.pth"
if best.exists():
    subprocess.run([sys.executable, "scripts/predict_test.py", "--config", tuned,
                    "--fold", str(FOLD), "--ckpt", str(best),
                    "--out", "/kaggle/working/submission.csv"], check=True)
print("outputs:", [str(p) for p in Path("/kaggle/working").iterdir()])